# 04B — Other IRS Form 990-Series: Form-Specific Feature Engineering

**Forms:** IRS990EZ, IRS990PF, IRS990T. **Source:** Notebook 03 comprehensive nested XML Parquet. **Purpose:** Build auditable, separate financial feature tables for forms that cannot safely share standard Form 990 field mappings.

### Design
1. Inventory actual forms, tax years, and XML tags before trusting mappings.
2. Extract only **explicit, form-specific fields**; keep missing as missing.
3. Save one partition per archive and form; reruns skip completed partitions.
4. Create harmonized features only when meanings are reasonably comparable.
5. Keep each form's data **separate from Notebook 04** and Notebook 02's significant-diversion label. No missing label becomes a negative.

**Important:** The mapping dictionary below is a **candidate mapping**, not a certification of all IRS schema revisions. This notebook produces coverage and source-tag audits and uses strict scoped lookups. Inspect low coverage and unexpected tag distributions before using the results for modeling.

**Default:** `RUN_EXTRACTION = False` for inspection first. Turn it on after reviewing the source-tag audit and the mapping definitions. Existing Notebook 04 outputs are never overwritten.

## 1. Setup and source inventory

In [ ]:
from pathlib import Path
from collections import Counter
import json, math
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

here = Path.cwd().resolve()
ROOT = next((q for q in (here, *here.parents) if (q / ".gitignore").exists()), None)
if ROOT is None:
    ROOT = here.parent if here.name == "notebooks" else here
SOURCE = ROOT / "data/staging/other_990_xml_partitions/full"
OUT = ROOT / "data/analytics/other_990_series_features"
PARTS = OUT / "partitions_v1"
REPORTS = OUT / "reports"
for d in (PARTS, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

FORMS = {"IRS990EZ": "irs990ez", "IRS990PF": "irs990pf", "IRS990T": "irs990t"}
FILES = [(form, p) for form, slug in FORMS.items()
         for p in sorted(SOURCE.glob(f"*/{slug}.parquet"))]
assert SOURCE.exists(), f"Missing Notebook 03 output: {SOURCE}"
assert FILES, f"No 990-EZ/PF/T Parquet files in {SOURCE}"

RUN_EXTRACTION = True  # Set True after reviewing source tag audit
OVERWRITE_PARTITIONS = False
BATCH_SIZE = 250
print("Repository:", ROOT)
print("Source files by form:", dict(Counter(form for form, _ in FILES)))
print("Total source partitions:", len(FILES))
display(pd.DataFrame([{
    "form": form, "archive": path.parent.name,
    "rows": pq.ParquetFile(path).metadata.num_rows
} for form, path in FILES]).groupby("form", as_index=False).agg(
    archives=("archive", "nunique"), filings=("rows", "sum")
))

## 2. Robust XML helpers and scoped lookups

In [ ]:
def local_name(tag):
    return str(tag).rsplit("}", 1)[-1].rsplit(":", 1)[-1]

def children_named(node, name):
    return [ch for ch in (node or {}).get("children", [])
            if local_name(ch.get("tag", "")) == name]

def one_child(node, name):
    matches = children_named(node, name)
    return matches[0] if len(matches) == 1 else None

def form_node(xml_tree_json, form_type):
    try:
        root = json.loads(xml_tree_json)
    except (TypeError, ValueError):
        return None, "invalid_json"
    data = one_child(root, "ReturnData")
    if data is None:
        return None, "return_data_missing_or_ambiguous"
    matches = children_named(data, form_type)
    if len(matches) != 1:
        return None, "form_missing" if not matches else "form_ambiguous"
    return matches[0], "ok"

def numeric(value):
    if value is None or isinstance(value, bool):
        return np.nan
    s = str(value).strip().replace(",", "").replace("$", "")
    if s.startswith("(") and s.endswith(")"):
        s = "-" + s[1:-1]
    try:
        v = float(s)
        return v if math.isfinite(v) else np.nan
    except (ValueError, TypeError):
        return np.nan

def scoped_text(node, path):
    # Strict direct-child path. Never search the entire tree for a repeated generic tag.
    current = node
    for tag in path[:-1]:
        found = children_named(current, tag)
        if len(found) != 1:
            return None, "missing_path" if not found else "ambiguous_path"
        current = found[0]
    found = children_named(current, path[-1])
    if len(found) != 1:
        return None, "missing_tag" if not found else "ambiguous_tag"
    value = found[0].get("text")
    if value is None or not str(value).strip():
        return None, "blank"
    return str(value).strip(), "ok"

def scoped_number(node, paths):
    # Only explicit form-specific alternatives. If two alternatives conflict, flag.
    candidates = []
    for path in paths:
        value, status = scoped_text(node, path)
        if status == "ok":
            num = numeric(value)
            if pd.notna(num):
                candidates.append((num, "/".join(path)))
    if not candidates:
        return np.nan, "missing", None
    values = {x[0] for x in candidates}
    if len(values) > 1:
        return np.nan, "conflicting_aliases", ";".join(x[1] for x in candidates)
    return candidates[0][0], "ok", candidates[0][1]

def iter_nodes(node, depth=0):
    if not isinstance(node, dict):
        return
    yield node, depth
    for child in node.get("children", []):
        yield from iter_nodes(child, depth + 1)

## 3. Inspect real source tags (do this before extraction)

A form-specific tag may differ by tax year or IRS schema version. The inventory reports **direct children of each main form**, plus tags containing key financial terms. It samples up to `SAMPLE_PER_FORM` filings across available archives. Increase the sample size if coverage looks unexpected.

This is **diagnostic**, not a substitute for reviewing official IRS schemas or sample XML. Only the scoped mappings in Section 4 are extracted.

In [ ]:
SAMPLE_PER_FORM = 60
inventory = []
examples = []
for form in FORMS:
    count = 0
    for candidate_form, path in FILES:
        if candidate_form != form or count >= SAMPLE_PER_FORM:
            continue
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(batch_size=min(25, SAMPLE_PER_FORM),
                                     columns=["filing_id", "tax_year", "xml_tree_json"]):
            for row in batch.to_pandas().to_dict("records"):
                node, status = form_node(row["xml_tree_json"], form)
                if node is None:
                    continue
                count += 1
                examples.append((form, str(row["filing_id"]), row["tax_year"]))
                for child in node.get("children", []):
                    inventory.append((form, local_name(child.get("tag", "")), str(row["filing_id"])))
                if count >= SAMPLE_PER_FORM:
                    break
            if count >= SAMPLE_PER_FORM:
                break
tags = pd.DataFrame(inventory, columns=["form", "direct_tag", "filing_id"])
tag_counts = tags.groupby(["form", "direct_tag"], as_index=False).agg(
    sampled_filings=("filing_id", "nunique")
).sort_values(["form", "sampled_filings"], ascending=[True, False])
for form in FORMS:
    print("\n", form, "— sampled filings:", sum(x[0] == form for x in examples))
    display(tag_counts.loc[tag_counts.form == form].head(75))
tag_counts.to_csv(REPORTS / "sampled_direct_child_tag_inventory.csv", index=False)

## 4. Consolidated, verified form-specific field maps

Mappings are based on the sample XML hierarchy inspected for 990-EZ, 990-PF, and 990-T. They are deliberately scoped to direct child paths; the same tag name in a different part of a form is **not** treated as equivalent. Coverage still needs to be audited across tax years and archives. PF fair-market-value assets and book-value assets are different measures; 990-T reports unrelated-business taxation, not an organization's total revenue. Missing or inapplicable fields remain null.

In [ ]:
# Verified form-specific direct-child XML paths, relative to each IRS form node.
# Missing fields remain null; do not substitute unrelated financial concepts.
MAP = {
    "IRS990EZ": {
        "ez_total_revenue": [["TotalRevenueAmt"]],
        "ez_total_expenses": [["TotalExpensesAmt"]],
        "ez_excess_revenue_over_expenses": [["ExcessOrDeficitForYearAmt"]],
        "ez_net_assets_boy": [["NetAssetsOrFundBalancesBOYAmt"]],
        "ez_net_assets_eoy": [["NetAssetsOrFundBalancesEOYAmt"]],
        "ez_total_assets_boy": [["Form990TotalAssetsGrp", "BOYAmt"]],
        "ez_total_assets_eoy": [["Form990TotalAssetsGrp", "EOYAmt"]],
        "ez_total_liabilities_boy": [["SumOfTotalLiabilitiesGrp", "BOYAmt"]],
        "ez_total_liabilities_eoy": [["SumOfTotalLiabilitiesGrp", "EOYAmt"]],
        "ez_contributions": [["ContributionsGiftsGrantsEtcAmt"]],
        "ez_program_service_revenue": [["ProgramServiceRevenueAmt"]],
        "ez_investment_income": [["InvestmentIncomeAmt"]],
        "ez_program_service_expenses": [["TotalProgramServiceExpensesAmt"]],
    },
    "IRS990PF": {
        "pf_total_revenue_books": [["AnalysisOfRevenueAndExpenses", "TotalRevAndExpnssAmt"]],
        "pf_total_expenses_books": [["AnalysisOfRevenueAndExpenses", "TotalExpensesRevAndExpnssAmt"]],
        "pf_excess_revenue_over_expenses": [["AnalysisOfRevenueAndExpenses", "ExcessRevenueOverExpensesAmt"]],
        "pf_net_investment_income": [["AnalysisOfRevenueAndExpenses", "NetInvestmentIncomeAmt"]],
        "pf_total_assets_boy": [["Form990PFBalanceSheetsGrp", "TotalAssetsBOYAmt"]],
        "pf_total_assets_book_eoy": [["Form990PFBalanceSheetsGrp", "TotalAssetsEOYAmt"]],
        "pf_total_assets_fmv_eoy": [["FMVAssetsEOYAmt"]],
        "pf_total_liabilities_eoy": [["Form990PFBalanceSheetsGrp", "TotalLiabilitiesEOYAmt"]],
        "pf_net_assets_boy": [["ChgInNetAssetsFundBalancesGrp", "TotNetAstOrFundBalancesBOYAmt"]],
        "pf_net_assets_eoy": [["ChgInNetAssetsFundBalancesGrp", "TotNetAstOrFundBalancesEOYAmt"]],
        "pf_grants_paid": [["AnalysisOfRevenueAndExpenses", "ContriPaidRevAndExpnssAmt"]],
        "pf_qualifying_distributions": [["PFQualifyingDistributionsGrp", "QualifyingDistributionsAmt"]],
    },
    "IRS990T": {
        "t_unrelated_business_taxable_income": [["TotalUBTIAmt"]],
        "t_ubti_computed": [["TotalUBTIComputedAmt"]],
        "t_ubti_before_nol_specific_deduction": [["TotalUBTIBeforeNOLSpecificAmt"]],
        "t_total_deductions": [["TotalDeductionAmt"]],
        "t_taxable_corporation_tax": [["TaxableCorporationAmt"]],
        "t_total_tax_computation": [["TotalTaxComputationAmt"]],
        "t_tax": [["TotalTaxAmt"]],
        "t_estimated_tax_payments": [["EstimatedTaxPaymentsAmt"]],
        "t_book_value_assets_eoy": [["BookValueAssetsEOYAmt"]],
    },
}
print("Verified XML mappings loaded:", {k: len(v) for k, v in MAP.items()})

## 5. Preview mappings on source filings before the full run

This table deliberately shows missing fields. If most values are missing, review Section 3's actual tags and amend the mappings rather than forcing zeros or using a global descendant search.

The preview displays `NaN` for fields belonging to other form types; that is expected.

In [ ]:
preview = []
for form in FORMS:
    path = next((p for typ, p in FILES if typ == form), None)
    if path is None:
        continue
    for batch in pq.ParquetFile(path).iter_batches(
        batch_size=10, columns=["filing_id", "tax_year", "xml_tree_json"]
    ):
        for row in batch.to_pandas().to_dict("records"):
            node, status = form_node(row["xml_tree_json"], form)
            rec = {"form_type": form, "filing_id": str(row["filing_id"]),
                   "tax_year": row["tax_year"], "xml_status": status}
            for field, paths in MAP[form].items():
                rec[field] = scoped_number(node, paths)[0] if node else np.nan
            preview.append(rec)
        break
preview_df = pd.DataFrame(preview)
display(preview_df)
print("Nonmissing mapped fields in sample (by form):")
for form in FORMS:
    sub = preview_df[preview_df.form_type == form]
    if len(sub):
        print(form, {field: int(sub[field].notna().sum()) for field in MAP[form]})

## 6. Extraction functions

Every record retains source provenance, field-level statuses, and the exact source tag path used. The XML tree is **not** copied into the output, keeping results compact.

In [ ]:
PROVENANCE = [
    "publication_year", "archive_name", "source_xml", "filing_id",
    "form_type", "ein", "tax_year", "tax_period_begin", "tax_period_end",
    "return_type", "organization_name"
]

def extract_other_form(row, form):
    result = {key: row.get(key) for key in PROVENANCE}
    result["filing_id"] = str(result["filing_id"])
    node, status = form_node(row.get("xml_tree_json"), form)
    result["main_form_status"] = status
    for field, paths in MAP[form].items():
        value, field_status, source_path = (
            scoped_number(node, paths) if node is not None
            else (np.nan, "main_form_missing", None)
        )
        result[field] = value
        result[field + "_status"] = field_status
        result[field + "_source_path"] = source_path
    return result

for form in FORMS:
    sub = preview_df.loc[preview_df.form_type == form]
    if not sub.empty:
        assert sub.filing_id.notna().all()
print("Extraction functions defined; preview identity checks passed.")

In [ ]:
# Validate representative known filings using each form's own source file.
# Never reuse a `batch` variable from another form's diagnostic cell.
known = {
    "IRS990EZ": {
        "filing_id": "202300109349200000",
        "expected": {
            "ez_total_revenue": 111640,
            "ez_total_expenses": 110548,
            "ez_total_assets_eoy": 182020,
            "ez_total_liabilities_eoy": 0,
            "ez_program_service_expenses": 102302,
        },
    },
    "IRS990PF": {
        "filing_id": "202300109349100000",
        "expected": {
            "pf_total_revenue_books": 118066,
            "pf_total_expenses_books": 70413,
            "pf_total_assets_boy": 1149393,
            "pf_total_assets_book_eoy": 1214045,
            "pf_total_liabilities_eoy": 0,
            "pf_net_assets_eoy": 1214045,
            "pf_grants_paid": 46162,
            "pf_qualifying_distributions": 49192,
        },
    },
    "IRS990T": {
        "filing_id": "202300109339300100",
        "expected": {
            "t_unrelated_business_taxable_income": 44638,
            "t_ubti_computed": 45638,
            "t_total_deductions": 1000,
            "t_tax": 9374,
        },
    },
}

for form, check in known.items():
    target = check["filing_id"]
    source_files = [p for f, p in FILES if f == form]
    found = None
    for source_file in source_files:
        pf = pq.ParquetFile(source_file)
        for b in pf.iter_batches(batch_size=BATCH_SIZE, columns=["filing_id", "xml_tree_json"]):
            for candidate in b.to_pandas().to_dict("records"):
                if str(candidate["filing_id"]) == target:
                    found = candidate
                    break
            if found is not None:
                break
        if found is not None:
            break
    assert found is not None, f"Known {form} filing not found: {target}"
    extracted = extract_other_form(found, form)
    assert extracted["main_form_status"] == "ok", (form, extracted["main_form_status"])
    for field, expected in check["expected"].items():
        actual = extracted[field]
        assert actual == expected, f"{form} {field}: expected {expected}, got {actual}"
    print(f"PASS {form}: {len(check['expected'])} checked fields")
print("All known-filing validation checks passed.")

## 7. Resumable archive-by-form extraction

**Start with `RUN_EXTRACTION=False` and review Sections 3–6, including known-filing checks.** Then set it to `True` and rerun this section. Only Form 990-EZ, PF and T files are read; full IRS990 files are skipped.

Writes one partition per archive/form. If you change `MAP`, **set `OVERWRITE_PARTITIONS=True`** to regenerate old partitions; otherwise stale outputs will be reused. Each write uses a temporary file and atomic replacement.

In [ ]:
if RUN_EXTRACTION:
    for i, (form, src) in enumerate(FILES, 1):
        archive = src.parent.name
        dest = PARTS / f"{archive}__{form.lower()}.parquet"
        if dest.exists() and not OVERWRITE_PARTITIONS:
            print(f"[{i}/{len(FILES)}] SKIP {archive} {form}", flush=True)
            continue
        records = []
        pf = pq.ParquetFile(src)
        for batch in pf.iter_batches(batch_size=BATCH_SIZE):
            for row in batch.to_pandas().to_dict("records"):
                records.append(extract_other_form(row, form))
        frame = pd.DataFrame.from_records(records)
        assert len(frame) == pf.metadata.num_rows, (src, "row count mismatch")
        assert frame.filing_id.is_unique, f"Duplicate filing ID in {src}"
        temp = dest.with_suffix(".tmp.parquet")
        frame.to_parquet(temp, index=False)
        temp.replace(dest)
        print(f"[{i}/{len(FILES)}] SAVED {archive} {form}: {len(frame):,}", flush=True)
else:
    print("Extraction is OFF. Review tag inventory and mapping preview, then set RUN_EXTRACTION=True.")

## 8. Load completed partitions and audit coverage

In [ ]:
expected = {f"{src.parent.name}__{form.lower()}.parquet" for form, src in FILES}
existing = {q.name for q in PARTS.glob("*.parquet")}
missing = expected - existing
if missing:
    print(f"Missing {len(missing)} output partitions; run Section 7 to create them.")
    print("Examples:", sorted(missing)[:5])
else:
    parts = sorted(PARTS / name for name in expected)
    data = pd.concat((pd.read_parquet(q) for q in parts), ignore_index=True)
    data["filing_id"] = data["filing_id"].astype(str)
    assert data.filing_id.is_unique, "Duplicate filing IDs across partitions"
    source_counts = Counter()
    for form, src in FILES:
        source_counts[form] += pq.ParquetFile(src).metadata.num_rows
    observed_counts = data.groupby("form_type").size().to_dict()
    assert all(observed_counts.get(form, 0) == n for form, n in source_counts.items()), (
        "Output counts do not match source counts"
    )
    audit_rows = []
    for form, fields in MAP.items():
        sub = data.loc[data.form_type == form]
        for field in fields:
            audit_rows.append({
                "form_type": form, "field": field, "filings": len(sub),
                "nonmissing": int(sub[field].notna().sum()),
                "coverage_pct": round(100 * sub[field].notna().mean(), 2) if len(sub) else np.nan,
                "status_counts": sub[field + "_status"].value_counts(dropna=False).to_dict()
            })
    audit_df = pd.DataFrame(audit_rows)
    display(audit_df[["form_type", "field", "filings", "nonmissing", "coverage_pct"]])
    audit_df.to_json(REPORTS / "field_coverage_and_status_v1.json",
                     orient="records", indent=2)
    print("Loaded:", f"{len(data):,}", "records")

## 9. Form-specific metrics and conservative harmonization

- Only 990-EZ revenue/expense fields are used to calculate an operating margin here.
- 990-PF uses its **own** book-value and fair-market-value concepts; no Form 990 program expense ratio is invented.
- 990-T income/tax measures are unrelated-business-specific and must not be interpreted as total organizational performance.
- Nulls remain null; ratios with zero denominators remain null.

In [ ]:
if "data" not in globals():
    print("No complete extracted dataset loaded yet. Run Sections 7–8 first.")
else:
    def safe_div(a, b):
        aa = pd.to_numeric(a, errors="coerce")
        bb = pd.to_numeric(b, errors="coerce")
        return aa.div(bb.where(bb.ne(0)))

    data["harmonized_total_revenue"] = np.nan
    data["harmonized_total_expenses"] = np.nan
    data["harmonized_assets_eoy"] = np.nan
    data["harmonized_liabilities_eoy"] = np.nan
    data["harmonization_scope"] = "not_comparable"

    ez = data.form_type.eq("IRS990EZ")
    data.loc[ez, "harmonized_total_revenue"] = data.loc[ez, "ez_total_revenue"]
    data.loc[ez, "harmonized_total_expenses"] = data.loc[ez, "ez_total_expenses"]
    data.loc[ez, "harmonized_assets_eoy"] = data.loc[ez, "ez_total_assets_eoy"]
    data.loc[ez, "harmonized_liabilities_eoy"] = data.loc[ez, "ez_total_liabilities_eoy"]
    data.loc[ez, "harmonization_scope"] = "990EZ_operating_summary_only"

    data["ez_operating_surplus"] = np.nan
    data["ez_operating_margin"] = np.nan
    data.loc[ez, "ez_operating_surplus"] = (
        data.loc[ez, "ez_total_revenue"] - data.loc[ez, "ez_total_expenses"]
    )
    data.loc[ez, "ez_operating_margin"] = safe_div(
        data.loc[ez, "ez_operating_surplus"], data.loc[ez, "ez_total_revenue"]
    )
    data["source_form_distinction_flag"] = True
    print("Form-specific features created; cross-form harmonization limited to 990-EZ.")

## 10. Validation and save separate datasets

This notebook intentionally does **not** join 990-EZ, 990-PF, or 990-T to the significant-diversion disclosure dataset. Some organizations may file multiple form types; **EIN + year is not a unique filing identity**. Any future entity-level combination needs a documented join policy and filing-type distinctions.

Review coverage and source-tag audits before modeling. Low coverage is a reason to refine mappings, **not** to fill zeros.

In [ ]:
if "data" not in globals():
    print("No outputs to save. Enable extraction and run Sections 7–9.")
else:
    assert data.filing_id.is_unique
    assert data.form_type.isin(FORMS).all()
    for form in FORMS:
        subset = data.loc[data.form_type == form].copy()
        if subset.empty:
            continue
        output = OUT / f"{form.lower()}_filing_features_v1.parquet"
        subset.to_parquet(output, index=False)
        print(f"Saved {form}: {len(subset):,} rows -> {output}")

    manifest = {
        "notebook": "04B_other_990_series_feature_engineering",
        "forms": sorted(FORMS),
        "total_filings": int(len(data)),
        "rows_by_form": {k: int(v) for k, v in data.form_type.value_counts().items()},
        "source_partitions": len(FILES),
        "coverage_report": str(REPORTS / "field_coverage_and_status_v1.json"),
        "warnings": [
            "Field mappings are candidate IRS e-file XML tags and must be checked against source coverage and schemas",
            "990-EZ/PF/T are not directly interchangeable with full Form 990",
            "No significant-diversion labels are assigned to these forms",
            "Unmatched fields are null, not zero",
            "Unusual values are review signals, not evidence of fraud"
        ]
    }
    (REPORTS / "04B_manifest_v1.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    print("04B saved. Review the field coverage audit before Notebook 05.")

## 11. Interpretation, caveats, and follow-up

**Data lineage:** IRS publication ZIPs → Notebook 03 complete XML tree Parquet → Notebook 04B per-form audited extraction → separate form-specific feature datasets.

**Known limitations:** IRS e-file schemas vary by form and tax year. The mapping dictionary must be verified against observed tags and authoritative schema definitions. The 990-T return is an unrelated-business tax return and should not be merged as if it were an ordinary nonprofit operating statement. PF's book-value and fair-market-value measures are distinct. Missing values are not assumed zero.

**Recommended follow-up:** Inspect the `sampled_direct_child_tag_inventory.csv` and `field_coverage_and_status_v1.json` reports, improve mappings where coverage is low, spot-check source XML, and only then decide which form-specific features are appropriate for Notebook 05. No verified fraud inference follows from these features alone.